# 2 — Population coding of object category in macaque IT

92 IT neurons recorded while 500 images were presented, each about ten
times. The images fall into four categories by stimulus code:

| Category   | Codes     | Images |
|------------|-----------|--------|
| Face       | 1–200     | 200    |
| Body       | 201–320   | 120    |
| Natural    | 321–390   | 70     |
| Artificial | 391–500   | 110    |

Eight analyses on the same data, in the order they build on each other:
single-neuron PSTHs, Fano factor, decoding accuracy over time, temporal
generalisation, mutual information, d-prime, and RSA.

**Data.** This notebook needs `dataVasati.mat`, which is not in the
repository — see the README. The figures in `assets/` were produced by these
cells, so the analysis can be read without re-running it.

**One thing to know before reading the code.** Trial order is *not* shared
between neurons: neuron 12's trial 40 and neuron 13's trial 40 are different
images. Every cell below resolves a stimulus against each neuron's own code
list. The original Fano cell did not, and that is written up in section 3.

In [ ]:
import os
from pathlib import Path

# The recordings are hundreds of megabytes and are not committed. Point
# SPIKE_DATA_DIR at wherever they live, or drop them in ./data/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = Path(os.environ.get('SPIKE_DATA_DIR') or ROOT / 'data')

if not DATA.is_dir():
    raise SystemExit(
        'No data directory at %s.\n'
        'Set SPIKE_DATA_DIR or create ./data/ — see the README.' % DATA)
print('data:', DATA)

RESULTS = ROOT / 'results'      # .mat / .npz written by the cells below
RESULTS.mkdir(exist_ok=True)
print('results:', RESULTS)


## 1 — Shared loaders

The original was nine standalone scripts, and seven of them carried their
own near-identical copy of the loader, the sliding-window PSTH and the
stimulus-code mapping. They are collected here once. Everything below is
the original analysis code with its private copies removed.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

MAT_FILE = DATA / 'dataVasati.mat'
FS = 1000                       # spike trains are binned at 1 kHz
START_MS, END_MS = 0, 550       # stimulus-locked epoch

CATEGORY_RANGES = [('Face', 1, 200), ('Body', 201, 320),
                   ('Natural', 321, 390), ('Artificial', 391, 500)]
CATEGORIES = [c for c, _, _ in CATEGORY_RANGES]

def map_to_class(code):
    """Stimulus code -> 0..3, or -1 if it falls outside every block."""
    for k, (_, lo, hi) in enumerate(CATEGORY_RANGES):
        if lo <= code <= hi:
            return k
    return -1

def load_per_neuron(start_ms=START_MS, end_ms=END_MS, fs=FS, path=MAT_FILE):
    """Per-neuron spike trains and the stimulus code of each of its trials.

    Returns (data_list, stim_list), both of length n_neurons. The two have
    to be indexed together: `stim_list[j][t]` is the code of neuron j's
    trial t, and trial order differs between neurons.
    """
    data_list, stim_list = [], []
    with h5py.File(path, 'r') as f:
        grp = f['SpikeTrain_it_all']
        s0, s1 = int(start_ms * fs / 1000), int(end_ms * fs / 1000)
        for i in range(grp['data'].shape[0]):
            cm = f[grp['cm'][i, 0]]
            if isinstance(cm, h5py.Dataset):
                stim = np.asarray(cm).astype(int).flatten()
            else:
                refs = np.asarray(cm['index']).flatten()
                stim = np.array([int(f[r][()][0]) for r in refs])
            arr = np.asarray(f[grp['data'][i, 0]])
            if arr.ndim == 1:
                arr = arr.reshape(1, -1)
            if arr.shape[0] != stim.size:
                arr = arr.T
            data_list.append(arr[:, s0:s1])
            stim_list.append(stim)
    return data_list, stim_list

def compute_psth(arr, win_ms=50, step_ms=5, fs=FS):
    """Sliding-window spike counts: (n_trials, n_bins) and bin centres."""
    win = int(win_ms * fs / 1000)
    step = int(step_ms * fs / 1000)
    starts = np.arange(0, arr.shape[1] - win + 1, step)
    feats = np.zeros((arr.shape[0], starts.size))
    for j, s0 in enumerate(starts):
        feats[:, j] = arr[:, s0:s0 + win].sum(axis=1)
    return feats, starts / fs * 1000 + win_ms / 2

def common_stimuli(stim_list):
    """Codes every neuron saw, and the smallest repeat count over those."""
    common = set(stim_list[0])
    for s in stim_list[1:]:
        common &= set(s)
    common = sorted(c for c in common if map_to_class(c) >= 0)
    min_rep = min((s == c).sum() for s in stim_list for c in common)
    return common, int(min_rep)

def build_population(data_list, stim_list, win_ms=50, step_ms=5, fs=FS):
    """Population responses, one sample per (stimulus, repeat).

    Returns X (n_samples, n_neurons, n_bins), y (n_samples,), bin centres.

    The repeat index is resolved against each neuron's own stimulus list, so
    row `r` of X holds every neuron's response to the *same* image. That is
    the one thing this function exists to guarantee — see section 3.
    """
    psths, times = [], None
    for arr in data_list:
        feats, times = compute_psth(arr, win_ms, step_ms, fs)
        psths.append(feats)

    common, min_rep = common_stimuli(stim_list)
    X = np.zeros((len(common) * min_rep, len(data_list), times.size))
    y = np.zeros(len(common) * min_rep, dtype=int)

    row = 0
    for stim in common:
        where = [np.where(s == stim)[0] for s in stim_list]
        for r in range(min_rep):
            for j, feats in enumerate(psths):
                X[row, j] = feats[where[j][r]]
            y[row] = map_to_class(stim)
            row += 1
    return X, y, times


## 2 — Single-neuron PSTHs by category

Sliding 50 ms window, 5 ms step, Gaussian-smoothed. The original drew one
figure per neuron — 92 near-identical panels — so three representative cells
are plotted instead; `PLOT_NEURONS = None` restores all of them.

In [ ]:
"""
Plot PSTH by stimulus class (face/body/natural/artificial) for each neuron,
with Gaussian smoothing.
"""
from scipy.ndimage import gaussian_filter1d

WIN_MS   = 50
STEP_MS  = 5
SMOOTH_SIGMA = 1.5

# The original drew one figure per neuron. Set to None for all 92.
PLOT_NEURONS = (1, 2, 6)
data_list, stim_list = load_per_neuron()
print(f'{len(data_list)} neurons, epoch {START_MS}-{END_MS} ms')

for neuron_idx, (data, stim) in enumerate(zip(data_list, stim_list), start=1):
    if PLOT_NEURONS and neuron_idx not in PLOT_NEURONS:
        continue
    feats, times = compute_psth(data, WIN_MS, STEP_MS, FS)
    plt.figure(figsize=(8,5))
    for cls_code, cls_name in enumerate(CATEGORIES):
        mask = (np.array([map_to_class(s) for s in stim]) == cls_code)
        if mask.sum() == 0:
            continue
        mean_psth = feats[mask].mean(axis=0)
        smooth_psth = gaussian_filter1d(mean_psth, sigma=SMOOTH_SIGMA)
        plt.plot(times, smooth_psth, label=cls_name)
    plt.title(f'Neuron {neuron_idx}: PSTH by class')
    plt.xlabel('Time (ms)')
    plt.ylabel(f'Spike count per {WIN_MS} ms')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## 3 — Fano factor, raw and mean-matched

Fano factor is variance over mean of the spike count, per neuron per bin. A
Poisson process sits at 1; IT responses are typically above it at baseline
and drop after stimulus onset. Because Fano is not independent of firing
rate, the second panel repeats the measurement on a rate-matched
subpopulation (ten quantiles of the first bin's mean count).

**Defect (structural, not measurable here).** The original read the stimulus
order from neuron 0 and then used it to index *every* neuron's trial array:

```python
data_list, stim_master = load_per_neuron(mat_file)   # stim_master from cm[0]
...
for r in idxs:                       # idxs from stim_master
    for j, data in enumerate(data_list):
        trial = data[r]              # neuron j indexed by neuron 0's order
```

Two cells later the decoding analysis looks the trial up per neuron
(`np.where(stim_list[j] == stim)`) and takes `min` over neurons to find the
repeat count — which only makes sense if trial counts and orders vary by
neuron. Both cannot be right. This notebook uses the per-neuron lookup
everywhere, so the counts being pooled into one variance are guaranteed to
come from the same image.

Note that the two disagree only about *which* trials get pooled. Fano
factor is computed across trials within a category, so the original was
still measuring a real quantity — just not the one it names.

In [ ]:
"""Fano factor over time, raw and rate-matched."""
from scipy.ndimage import gaussian_filter1d
def compute_fano(X):
    """
    Vectorized Fano computation.
    X: (n_samples, N, T), as returned by build_population
    Returns:
      FF: (N, T)
      mean_counts: (N, T)
    """
    mean_counts = X.mean(axis=0)
    var_counts  = X.var(axis=0, ddof=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        FF = var_counts / mean_counts
    return FF, mean_counts
def mean_match_ff(FF, mean_counts, n_quantiles=10, seed=0):
    """
    Control for mean‐rate differences across bins.
    FF, mean_counts: (N, T)
    Returns:
      FF_mm: (T,) mean‐matched Fano vs time
    """
    rng = np.random.RandomState(seed)
    N, T = FF.shape
    ref = mean_counts[:, 0]
    bins = np.quantile(ref, np.linspace(0,1,n_quantiles+1))
    hist_ref, _ = np.histogram(ref, bins=bins)
    FF_mm = np.full(T, np.nan)
    for k in range(T):
        vals = mean_counts[:, k]
        idxs = []
        for b in range(n_quantiles):
            candidates = np.where((vals >= bins[b]) & (vals < bins[b+1]))[0]
            if candidates.size:
                rng.shuffle(candidates)
                take = min(hist_ref[b], candidates.size)
                idxs.extend(candidates[:take])
        if idxs:
            FF_mm[k] = np.nanmean(FF[idxs, k])
    return FF_mm
WIN_MS, STEP_MS = 15, 4

data_list, stim_list = load_per_neuron()
X, y, times = build_population(data_list, stim_list, WIN_MS, STEP_MS)
print(f'X {X.shape}   {times.size} bins of {WIN_MS} ms')

cat_names = CATEGORIES
n_slices  = X.shape[2]
FF_mean_all = np.zeros((len(cat_names), n_slices))
FF_mm_all   = np.zeros((len(cat_names), n_slices))
for i, cname in enumerate(cat_names):
    mask = (y == i)
    Xc = X[mask]
    FFc, mean_counts = compute_fano(Xc)
    FF_mean_all[i] = np.nanmean(FFc, axis=0)
    FF_mm_all[i]   = mean_match_ff(FFc, mean_counts, n_quantiles=10, seed=42)
sigma = 2
FF_mean_sm = gaussian_filter1d(FF_mean_all, sigma=sigma, axis=1)
FF_mm_sm   = gaussian_filter1d(FF_mm_all,   sigma=sigma, axis=1)
plt.figure(figsize=(6,4))
for i, cname in enumerate(cat_names):
    plt.plot(times, FF_mean_sm[i], '-o', label=cname)
plt.xlabel('Time (ms)')
plt.ylabel('Fano Factor (smoothed)')
plt.title(f'Gaussian Smoothed FF (σ={sigma})')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.figure(figsize=(6,4))
for i, cname in enumerate(cat_names):
    plt.plot(times, FF_mm_sm[i], '-s', label=cname)
plt.xlabel('Time (ms)')
plt.ylabel('Fano Factor (mean-matched, smoothed)')
plt.title(f'Gaussian Smoothed Mean-Matched FF (σ={sigma})')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 4 — Decoding category from the population, over time

One RBF SVM per time bin, one-vs-one over the four categories, five-fold
stratified CV. Two things were fixed here.

**Defect: the scaler saw the test folds.** The original standardised the
whole time slice and then cross-validated the standardised matrix:

```python
scaler = StandardScaler()
Xi = scaler.fit_transform(X[:, :, ts])        # all 5000 rows
y_pred = cross_val_predict(clf, Xi, y, cv=cv) # then split into folds
```

Every fold's held-out rows had contributed their mean and variance to the
transform. Wrapping the scaler in a `Pipeline` refits it inside each fold.
With 5000 samples and 92 features the leak is small, but it is free to fix
and it is the kind of thing a reviewer looks for first.

**Defect: chance was drawn at 25%.** The four categories hold 200, 120, 70
and 110 images, so a classifier that always answered *face* scores 40%. The
dashed line is the majority-class rate.

In [ ]:
"""Population decoding of category, one classifier per time bin."""
from sklearn.svm import SVC
from sklearn.multiclass import OneVsOneClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
from scipy.io import savemat
def classify_over_time(X, y):
    n_slices = X.shape[2]
    accuracy = np.zeros(n_slices)
    recall = np.zeros((len(CATEGORIES), n_slices))
    # DEFECT. The original standardised the whole slice and then
    # cross-validated the result, so each fold's test rows had contributed
    # their mean and variance to the transform. In a Pipeline the scaler is
    # refitted inside every fold.
    for ts in range(n_slices):
        Xi = X[:, :, ts]
        clf = make_pipeline(
            StandardScaler(),
            OneVsOneClassifier(SVC(kernel='rbf', C=1.0, gamma='scale',
                                   class_weight='balanced')))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        y_pred = cross_val_predict(clf, Xi, y, cv=cv, n_jobs=-1)
        accuracy[ts] = np.mean(y_pred == y) * 100
        cm = confusion_matrix(y, y_pred, labels=[0,1,2,3])
        recall[:, ts] = np.diag(cm) / np.sum(cm, axis=1) * 100
    return accuracy, recall
WIN_MS, STEP_MS = 50, 5

data_list, stim_list = load_per_neuron()
X, y, times = build_population(data_list, stim_list, WIN_MS, STEP_MS)

# DEFECT. Chance was drawn at 25%. The categories hold 200/120/70/110
# images, so always answering "Face" already scores 40%.
CHANCE = 100.0 * np.bincount(y).max() / y.size
print(f'X {X.shape}   majority-class baseline {CHANCE:.1f}%  (1/4 = 25.0%)')

accuracy, recall = classify_over_time(X, y)

plt.figure(figsize=(7, 4))
plt.plot(times, accuracy, '-o', ms=4)
plt.axhline(CHANCE, ls='--', c='k', lw=0.8,
            label=f'majority class ({CHANCE:.0f}%)')
plt.xlabel('Time (ms)'); plt.ylabel('Accuracy (%)')
plt.title('Population decoding accuracy')
plt.legend(); plt.grid(True); plt.tight_layout()

plt.figure(figsize=(7, 4))
for i, name in enumerate(CATEGORIES):
    plt.plot(times, recall[i], label=name)
plt.xlabel('Time (ms)'); plt.ylabel('Recall (%)')
plt.title('Per-category recall')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()

savemat(RESULTS / 'pop_results.mat',
        {'times': times, 'accuracy': accuracy, 'recall': recall})
print(f'Max accuracy {accuracy.max():.2f}% at '
      f'{times[np.argmax(accuracy)]:.1f} ms')


## 5 — Temporal generalisation (time–time decoding)

Train a linear SVM on one time bin, test it on every other bin. The
diagonal is ordinary decoding; off-diagonal structure says the population
code is stable across those two latencies rather than being re-encoded.
Here each image is averaged over its ten repeats first, so there are 500
samples rather than 5000, and significance comes from 500 label
permutations with cluster-mass correction.

**Defect: every time label was 100 ms early.** The PSTH bin offsets are
counted from the start of the *cropped* epoch, which begins at
`EPOCH_START_MS = 100`, but the offset was never added back:

```python
bins_idx = np.arange(psths0, n_time - win_pts + 1, step_pts)
times    = bins_idx / FS * 1000 + WIN_MS/2      # missing EPOCH_START_MS
```

The print statement on the next line gives it away — it adds
`EPOCH_START_MS` to the end of the range but not to the start. The published
figure is labelled 100–320 ms and is really 200–420 ms, which matters
because that is the difference between "category information appears before
100 ms" and "appears after 200 ms". The corrected axis puts the onset where
the decoding curve in section 4 puts it.

`N_PERM = 500` takes roughly 40 minutes on eight cores.

In [ ]:
"""Temporal generalisation: train on one time bin, test on all of them.

- 60 ms PSTH window, 5 ms step, epoch 100-450 ms
- each image averaged over its ~10 repeats first, so 500 samples
- linear SVM, one-vs-one, 5-fold stratified CV
- 500 label permutations with cluster-mass correction
"""
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsOneClassifier
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed, parallel_backend
from scipy.ndimage import label
from tqdm import tqdm
from scipy.io import savemat
EPOCH_START_MS, EPOCH_END_MS = 100, 450  
WIN_MS , STEP_MS  = 60 , 5
PSTH_START_MS = 50
CV_FOLDS    = 5
N_PERM      = 500      # ~40 min on 8 cores; 50 for a quick pass
PLOT_START_MS = 200    # crop for the figure, see DEFECT below

# Set from the label counts once `y` exists, not 1/n_classes.
CHANCE_ACC  = None
def cluster_perm(obs, null, chance, alpha=0.05):
    diff_obs  = obs  - chance
    diff_null = null - chance
    thr = np.percentile(diff_null.ravel(), 100 * (1 - alpha))
    supra, n  = label(diff_obs > thr, structure=np.ones((3, 3)))
    masses = [diff_obs[supra == i].sum() for i in range(1, n + 1)]
    maxnull = []
    for m in diff_null:
        s2, n2 = label(m > thr, structure=np.ones((3, 3)))
        maxnull.append(max([m[s2 == i].sum() for i in range(1, n2 + 1)] or [0]))
    crit = np.percentile(maxnull, 100 * (1 - alpha))
    sig  = np.zeros_like(obs, bool)
    for i, mass in enumerate(masses, 1):
        if mass > crit:
            sig[supra == i] = True
    return sig
dataN, stimN = load_per_neuron(EPOCH_START_MS, EPOCH_END_MS)
N_neu = len(dataN)
print(f'  loaded {N_neu} neurons, epoch {EPOCH_START_MS}-{EPOCH_END_MS} ms')
images = set(stimN[0])
for st in stimN[1:]:
    images &= set(st)
images = sorted(images)
min_rep = min((sl==img).sum() for sl in stimN for img in images)
print(f'  {len(images)} common stimuli, min repetition = {min_rep}')
win_pts  = int(WIN_MS  * FS/1000)
step_pts = int(STEP_MS * FS/1000)
psths0   = int(PSTH_START_MS * FS/1000)
n_time   = dataN[0].shape[1]
bins_idx = np.arange(psths0, n_time - win_pts + 1, step_pts)
# DEFECT. `bins_idx` counts samples from the start of the *cropped*
# epoch, so EPOCH_START_MS has to be added back. Without it every label is
# 100 ms early — the published figure reads 100-320 ms and is really
# 200-420 ms. The print below is what gave it away: it added the offset to
# the end of the range but not to the start.
times    = EPOCH_START_MS + bins_idx / FS * 1000 + WIN_MS/2
n_bins   = bins_idx.size
print(f'  {n_bins} PSTH bins spanning {times[0]:.0f}-{times[-1]:.0f} ms')
N_img = len(images)
X = np.zeros((N_img, N_neu, n_bins))
y = np.zeros(N_img, int)
for k, img in enumerate(images):
    y[k] = map_to_class(img)
    for n in range(N_neu):
        trials = dataN[n]
        stims   = stimN[n]
        idxs    = np.where(stims==img)[0][:min_rep]
        psth_avg = trials[idxs].mean(axis=0)
        for b, start in enumerate(bins_idx):
            X[k, n, b] = psth_avg[start:start+win_pts].sum()
print('  feature matrix X shape =', X.shape)

CHANCE_ACC = 100.0 * np.bincount(y).max() / y.size
print(f'  majority-class baseline = {CHANCE_ACC:.1f}%   (1/4 = 25.0%)')
cv = list(StratifiedKFold(CV_FOLDS, shuffle=True, random_state=0)
          .split(np.zeros(N_img), y))
def decode_train_bin(b):
    acc = np.zeros(n_bins)
    for tr, te in cv:
        clf = OneVsOneClassifier(
            LinearSVC(dual=False, C=1.0, class_weight='balanced', max_iter=5000)
        )
        clf.fit(X[tr, :, b], y[tr])
        flat = X[te].transpose(2,0,1).reshape(n_bins * len(te), N_neu)
        pr   = clf.predict(flat).reshape(n_bins, len(te))
        acc += (pr == y[te]).mean(axis=1)
    return acc / len(cv) * 100
with parallel_backend('threading'):
    dec_mat = np.vstack(Parallel(n_jobs=-1)(
        delayed(decode_train_bin)(b) for b in range(n_bins)
    ))
null = np.zeros((N_PERM, n_bins, n_bins))
rng  = np.random.RandomState(0)
def decode_perm(y_perm, b):
    acc = np.zeros(n_bins)
    for tr, te in cv:
        clf = OneVsOneClassifier(
            LinearSVC(dual=False, C=1.0, class_weight='balanced', max_iter=5000)
        )
        clf.fit(X[tr, :, b], y_perm[tr])
        flat = X[te].transpose(2,0,1).reshape(n_bins * len(te), N_neu)
        pr   = clf.predict(flat).reshape(n_bins, len(te))
        acc += (pr == y_perm[te]).mean(axis=1)
    return acc / len(cv) * 100
for p in tqdm(range(N_PERM), desc='Permuting'):
    yp = rng.permutation(y)
    null[p] = Parallel(n_jobs=-1, backend='threading')(
        delayed(decode_perm)(yp, b) for b in range(n_bins)
    )
sig = cluster_perm(dec_mat, null, alpha=0.05, chance=CHANCE_ACC)
# With the offset restored the first bin is already past 100 ms, so the
# original crop would no longer trim anything.
i0 = np.searchsorted(times, PLOT_START_MS)
times_plot = times[i0:]
dec_plot   = dec_mat[i0:, i0:]
sig_plot   = sig[i0:, i0:]
plt.figure(figsize=(6,5))
im = plt.imshow(dec_plot, origin='lower', aspect='auto',
           extent=[times_plot[0], times_plot[-1],
                   times_plot[0], times_plot[-1]],
           vmin=CHANCE_ACC, vmax=dec_mat.max(), cmap='viridis')
plt.colorbar(im, label='Accuracy (%)')
plt.contour(times_plot, times_plot, sig_plot, levels=[0.5],
            colors='white', linewidths=2)
plt.xlabel('Test time (ms)')
plt.ylabel('Train time (ms)')
plt.title(f'Time–Time decoding | window {WIN_MS} ms, step {STEP_MS} ms')
plt.tight_layout()
plt.show()
savemat(RESULTS / 'pop_TTdecoding_60ms_step5.mat',
        dict(times=times_plot, decoding=dec_plot, sig=sig_plot))
print('Done.')


## 6 — Mutual information over time

Per-neuron mutual information between binned spike count and category,
summed over the population, with a permutation null.

**Defect: the p-values could not clear their own threshold.** The original
estimated `p = mean(null >= obs)`, which returns exactly 0 when no
permutation beats the observed value, and then compared it against a
Bonferroni threshold of `0.05 / 51 ≈ 9.8e-4`. With 1000 permutations the
finest resolvable p-value is 1e-3 — the same order as the threshold — so
"significant" and "we ran out of permutations" were indistinguishable. The
add-one estimator `(1 + count) / (1 + n_perm)` at least reports the bound
honestly; resolving a Bonferroni-corrected effect properly needs `N_PERMS`
in the tens of thousands, which is noted rather than run.

In [ ]:
"""
Mutual Information Analysis Across Time with Permutation Test and Bonferroni Correction
--------------------------------------------------------------------------------
Dataset: dataVasati.mat (SpikeTrain_it_all)
PSTH Window = 50 ms, Step = 10 ms (overlapping)
Number of Permutations = 1000
Each neuron has its own trial labels (stimulus codes). We first organize data by stimulus.
"""
from sklearn.feature_selection import mutual_info_classif
from joblib import Parallel, delayed
from tqdm import tqdm
from scipy.io import savemat
WIN_MS   = 50
STEP_MS  = 10
N_PERMS  = 1000
ALPHA    = 0.05
N_JOBS   = -1
def mi_sum(Xb, y):
    """Summed per-neuron mutual information in one time bin.

    `discrete_features=True` is right — the features are spike counts — but
    it wants integers, and the shared builder returns float.
    """
    return mutual_info_classif(Xb.astype(int), y, discrete_features=True,
                               random_state=0).sum()
def perm_mi(seed, X, y, n_bins):
    """Compute MI for permuted data."""
    rng = np.random.RandomState(seed)
    y_perm = rng.permutation(y)
    out = np.zeros(n_bins)
    for b in range(n_bins):
        out[b] = mi_sum(X[:, :, b], y_perm)
    return out
data_list, stim_list = load_per_neuron()
X, y, times = build_population(data_list, stim_list, WIN_MS, STEP_MS)
n_bins = X.shape[2]
print(f'{X.shape[0]} trials, {X.shape[1]} neurons, {n_bins} bins')

obs_mi = np.array([mi_sum(X[:, :, b], y) for b in range(n_bins)])

null = np.stack(Parallel(n_jobs=N_JOBS)(
    delayed(perm_mi)(seed, X, y, n_bins)
    for seed in tqdm(range(N_PERMS), desc='Permuting')), axis=0)

# DEFECT. `mean(null >= obs)` returns exactly 0 when no permutation beats
# the observation, which reads as p = 0 and really means p < 1/N_PERMS. The
# add-one estimator bounds it at 1/(N_PERMS+1) = 1.0e-3 — and note that the
# Bonferroni threshold below is 0.05/51 = 9.8e-4, the same order. Resolving
# a corrected effect properly needs N_PERMS in the tens of thousands.
pvals = (1 + (null >= obs_mi[None, :]).sum(axis=0)) / (1 + N_PERMS)
alpha_bonf = ALPHA / n_bins
sig = pvals < alpha_bonf
print(f'permutation resolution = {1 / (1 + N_PERMS):.1e}, '
      f'Bonferroni threshold = {alpha_bonf:.1e}')

plt.figure(figsize=(9, 4))
plt.plot(times, obs_mi, '-o', lw=2, ms=4, label='MI (observed)', color='C0')
plt.plot(times, null.mean(axis=0), lw=1, c='grey', label='permutation mean')
plt.fill_between(times, 0, obs_mi.max(), where=sig, color='red', alpha=0.15,
                 label=f'p < {ALPHA}/{n_bins} (Bonferroni)')
plt.xlabel('Time (ms)'); plt.ylabel('Mutual information (bits)')
plt.title('Summed population mutual information over time')
plt.legend(); plt.grid(True, ls='--', alpha=0.6); plt.tight_layout()
plt.show()

savemat(RESULTS / 'mi_results.mat',
        {'times': times, 'obs_mi': obs_mi, 'null': null,
         'pvals': pvals, 'sig': sig})

if sig.any():
    print(f'significant bins: {times[sig]}')
else:
    print('no bins survive Bonferroni correction')


## 7 — d-prime between category pairs

Single-neuron separability of spike counts in the 100–300 ms window, for
each of the six category pairs, averaged over neurons with SEM. The original
labelled the axis "d′ (bits)" — d-prime is a standardised mean difference
and is dimensionless; bits belong to section 6.

In [ ]:
"""d-prime between category pairs.

Spike counts in the 100-300 ms window, six pairwise comparisons,
d' = (m1 - m2) / sqrt((s1^2 + s2^2) / 2), averaged over neurons with SEM.
d-prime is a standardised mean difference, so it is dimensionless — the
original labelled the axis "bits", which belongs to the MI section.
"""
import itertools

WIN_START = 100
WIN_END   = 300
data_list, stim_list = load_per_neuron()
n_neurons = len(data_list)
pairs = list(itertools.combinations(CATEGORIES, 2))
dprime = np.zeros((n_neurons, len(pairs)))
w0 = WIN_START  - START_MS
w1 = WIN_END    - START_MS
for i in range(n_neurons):
    data = data_list[i]
    stim = stim_list[i]
    counts = data[:, w0:w1].sum(axis=1)
    cls = np.array([map_to_class(s) for s in stim])
    grp_counts = {cat: counts[cls == k] for k, cat in enumerate(CATEGORIES)}
    for j,(c1,c2) in enumerate(pairs):
        x1 = grp_counts[c1]
        x2 = grp_counts[c2]
        if x1.size<2 or x2.size<2:
            dprime[i,j] = np.nan
            continue
        mu1, mu2 = x1.mean(), x2.mean()
        s1, s2   = x1.std(ddof=1), x2.std(ddof=1)
        denom    = np.sqrt(0.5*(s1**2 + s2**2))
        dprime[i,j] = (mu1 - mu2)/denom if denom>0 else 0
mean_dp = np.nanmean(dprime, axis=0)
sem_dp  = np.nanstd(dprime, axis=0, ddof=1)/np.sqrt(np.sum(~np.isnan(dprime),axis=0))
plt.figure(figsize=(8, 5))
x = np.arange(len(pairs))
plt.bar(x, mean_dp, yerr=sem_dp, capsize=5, color='C0')
plt.xticks(x, [f'{a} vs {b}' for a, b in pairs], rotation=45, ha='right')
plt.axhline(0, c='k', lw=0.5)
plt.ylabel("d'")
plt.title(f'Single-neuron d-prime by category pair '
          f'({WIN_START}-{WIN_END} ms, n = {n_neurons})')
plt.grid(axis='y', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

for (a, b), m, s in zip(pairs, mean_dp, sem_dp):
    print(f'{a:>10} vs {b:<10}  d\' = {m:+.3f} +/- {s:.3f}')


## 8 — Representational similarity, over time

For each time bin, build the 500×500 neural RDM (1 − correlation between
population response patterns) and compare it to a binary category-model RDM
with Kendall's tau. The rise of tau tracks when the population geometry
becomes category-like.

The original computed p-values for every bin and then never used them. With
111 bins an uncorrected 0.05 says nothing, so the shaded band here is
Bonferroni-corrected.

In [ ]:
"""
Time‐resolved RDM analysis with Kendall’s tau
"""
from scipy.stats import kendalltau
from tqdm import tqdm
WIN_MS     =  50
STEP_MS    =   5
ALPHA      = 0.05
data_list, stim_list = load_per_neuron()
n_neu = len(data_list)
feats_list, times = [], None
for arr in data_list:
    fe, t = compute_psth(arr, WIN_MS, STEP_MS, FS)
    feats_list.append(fe)
    times = t
n_bins   = feats_list[0].shape[1]
common = set(stim_list[0])
for sl in stim_list[1:]:
    common &= set(sl)
common = sorted(common)
S = len(common)
labels = np.array([map_to_class(s) for s in common])
gt = np.zeros((S,S), int)
for i in range(S):
    for j in range(S):
        gt[i,j] = 0 if labels[i]==labels[j] else 1
triu_idx = np.triu_indices(S, k=1)
gt_flat  = gt[triu_idx]
taus = np.zeros(n_bins)
ps   = np.zeros(n_bins)
for b in tqdm(range(n_bins), desc="RDM & τ"):
    patt = np.zeros((S, n_neu), float)
    for i, s in enumerate(common):
        for j in range(n_neu):
            idxs = np.where(stim_list[j]==s)[0]
            patt[i,j] = feats_list[j][idxs, b].mean()
    corr = np.corrcoef(patt)
    rdm  = 1 - corr
    neu_flat = rdm[triu_idx]
    tau, p   = kendalltau(neu_flat, gt_flat)
    taus[b]  = tau
    ps[b]    = p
# The original computed `ps` for all 111 bins and never used them.
sig = ps < ALPHA / n_bins

plt.figure(figsize=(8, 3.5))
plt.plot(times, taus, '-o', ms=4)
plt.axhline(0, color='k', lw=0.5)
plt.fill_between(times, 0, taus.max(), where=sig, color='red', alpha=0.15,
                 label=f'p < {ALPHA}/{n_bins} (Bonferroni)')
plt.xlabel('Time (ms)')
plt.ylabel("Kendall's tau")
plt.title('Time-resolved RSA: neural RDM vs category model')
plt.legend(); plt.grid(True, ls='--', alpha=0.5); plt.tight_layout()
plt.show()

b_max = int(np.argmax(taus))
print(f'peak tau = {taus[b_max]:.3f} at {times[b_max]:.0f} ms '
      f'(p = {ps[b_max]:.2e})')

np.savez(RESULTS / 'rdm_time_course.npz', times=times, tau=taus, pval=ps)


## 9 — The neural RDM at one latency

The neural RDM beside the category model it is being compared against.
Stimuli are in code order, so the category blocks are contiguous and a
category-like representation shows up as visible block structure.

The original selected the bin by hardcoded index (`w = 10`). That is 75 ms,
which section 4 shows is before category information is decodable — so this
RDM looking close to structureless is the expected result, not a bug. The
index is expressed as a latency here so the figure can be read.

In [ ]:
"""Neural RDM at a single latency, beside the category-model RDM."""
from scipy.spatial.distance import pdist, squareform

WIN_MS, STEP_MS = 50, 5
data_list, stim_list = load_per_neuron()
n_neu = len(data_list)
psth_list = []
for arr in data_list:
    feats, times = compute_psth(arr, WIN_MS, STEP_MS, FS)
    psth_list.append(feats)
n_bins   = psth_list[0].shape[1]
common = set(stim_list[0])
for sl in stim_list[1:]:
    common &= set(sl)
common = sorted(common)
n_stim  = len(common)
min_rep = min((sl==s).sum() for sl in stim_list for s in common)
# The original hardcoded bin index 10. Naming the latency makes it clear
# which moment the RDM describes — and that 75 ms is early, before category
# information is decodable (section 4), which is why this RDM is close to
# structureless.
RDM_TIME_MS = 75
w = int(np.argmin(np.abs(times - RDM_TIME_MS)))
patterns = np.zeros((n_stim, n_neu), float)
for i_s, stim in enumerate(common):
    vals = []
    for j in range(n_neu):
        trials_idx = np.where(stim_list[j] == stim)[0][:min_rep]
        vals.append(psth_list[j][trials_idx, w].mean())
    patterns[i_s] = vals
def one_minus_corr(u,v):
    return 1 - np.corrcoef(u,v)[0,1]
rdm_neural = squareform(pdist(patterns, metric=one_minus_corr))
classes = np.array([map_to_class(s) for s in common])
gt_rdm = (classes[:,None] != classes[None,:]).astype(float)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
im1 = ax1.imshow(rdm_neural, vmin=0, vmax=1, cmap='viridis')
ax1.set_title(f'Neural RDM @ {times[w]:.0f} ms')
fig.colorbar(im1, ax=ax1, fraction=0.046, label='1 - corr')

im2 = ax2.imshow(gt_rdm, vmin=0, vmax=1, cmap='viridis')
ax2.set_title('Category-model RDM')
fig.colorbar(im2, ax=ax2, fraction=0.046)

# `common` is in stimulus-code order and the categories are contiguous
# code ranges, so the class boundaries are where `classes` changes.
bounds = 0.5 + np.flatnonzero(np.diff(classes))
for ax in (ax1, ax2):
    ax.set_xlabel('Stimulus')
    ax.set_ylabel('Stimulus')
    for b in bounds:
        ax.axhline(b, c='w', lw=0.6)
        ax.axvline(b, c='w', lw=0.6)

plt.tight_layout()
plt.show()


## 10 — RDMs across the response window

**Defect: the grid did not show the window in its title.** The bin filter
kept every bin whose centre was a multiple of 5 ms between 150 and 300 ms —
31 bins — and then `zip(axes, w_inds)` silently truncated to the 16
available panels. The figure was titled "150–300 ms (10 ms bins)" and
actually showed 150–225 ms in 5 ms steps. Asking for 10 ms steps, as the
title says, gives exactly 16 bins and covers the window.

In [ ]:
"""Neural RDMs across the response window, one panel per latency."""
from scipy.spatial.distance import pdist, squareform

WIN_MS, STEP_MS = 50, 5
GRID_START_MS, GRID_END_MS, GRID_STEP_MS = 150, 300, 10
data_list, stim_list = load_per_neuron()
n_neu = len(data_list)
psth_list = []
for arr in data_list:
    feats, times = compute_psth(arr, WIN_MS, STEP_MS, FS)
    psth_list.append(feats)
common = set(stim_list[0])
for sl in stim_list[1:]:
    common &= set(sl)
common = sorted(common)
n_stim  = len(common)
min_rep = min((sl == s).sum() for sl in stim_list for s in common)
# DEFECT. The original kept every bin on a 5 ms grid between 150 and 300 ms
# — 31 of them — and then `zip(axes, w_inds)` silently truncated to the 16
# panels available. The figure was titled "150-300 ms (10 ms bins)" and
# actually showed 150-225 ms. A 10 ms grid gives exactly 16 bins and covers
# the window the title claims.
w_inds = [i for i, t in enumerate(times)
          if GRID_START_MS <= t <= GRID_END_MS
          and (t - GRID_START_MS) % GRID_STEP_MS == 0]
assert len(w_inds) == 16, len(w_inds)

def one_minus_corr(u, v):
    return 1 - np.corrcoef(u, v)[0, 1]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, w in zip(axes.flatten(), w_inds):
    patterns = np.zeros((n_stim, n_neu))
    for i_s, stim in enumerate(common):
        for j in range(n_neu):
            idx = np.where(stim_list[j] == stim)[0][:min_rep]
            patterns[i_s, j] = psth_list[j][idx, w].mean()
    rdm = squareform(pdist(patterns, metric=one_minus_corr))
    im = ax.imshow(rdm, vmin=0, vmax=1, cmap='viridis')
    ax.set_title(f'{times[w]:.0f} ms')
    ax.set_xticks([]); ax.set_yticks([])

cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01)
cbar.set_label('1 - corr')
plt.suptitle(f'Neural RDMs, {GRID_START_MS}-{GRID_END_MS} ms '
             f'in {GRID_STEP_MS} ms steps')
plt.show()
